
# Phase 3 — Model Development (Updated)

This notebook develops two classification systems:

**Model A — Visit Risk Classification**
Predict whether a hospital visit is Low, Medium, or High risk.

**Model B — Claim Outcome Classification**
Predict whether an insurance claim will be Paid, Pending, or Rejected.

Multiple algorithms are evaluated to identify the best performing model.


## Step 1 — Import Libraries

In [1]:

import pandas as pd
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib


## Step 2 — Load Modeling Dataset

In [2]:

df = pd.read_csv("F:\AI ML\capstone\phase 2\model_table.csv")

print(df.shape)
df.head()


(25000, 26)


<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\praka\AppData\Local\Temp\ipykernel_18868\4055138432.py:1: SyntaxWarning: invalid escape sequence '\A'
  df = pd.read_csv("F:\AI ML\capstone\phase 2\model_table.csv")


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,visit_frequency,avg_los_per_patient,provider_rejection_rate,days_since_registration,visit_month,visit_dayofweek
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,2,3.725000,0.148655,65,10,5
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,4,32.025000,0.156915,-206,4,6
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,4,20.542500,0.149678,9,7,6
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,7,28.165714,0.152480,-62,11,2
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,5,22.988000,0.149678,0,3,5


## Step 3 — Time-based Train Test Split

In [3]:

df["visit_date"] = pd.to_datetime(df["visit_date"])
df = df.sort_values("visit_date")

split = int(len(df)*0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:]

train_df = train_df.copy()
test_df = test_df.copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)


Train: (20000, 26)
Test: (5000, 26)


## MODEL A — Visit Risk Classification

### Step 4 — Feature Selection

In [4]:

risk_features = [
    "age",
    "chronic_flag",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "days_since_registration",
    "visit_month",
    "visit_dayofweek"
]

X_train = train_df[risk_features].fillna(0)
X_test = test_df[risk_features].fillna(0)

y_train = train_df["risk_score"]
y_test = test_df["risk_score"]


### Step 5 — Train Multiple Models

In [5]:

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=10),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingClassifier()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    
    results.append([name, acc, f1])
    
    print(f"\n{name}")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

results_df = pd.DataFrame(results, columns=["Model","Accuracy","Macro_F1"])
results_df


c:\Users\praka\.virtualenvs\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\praka\.virtualenvs\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\praka\.virtualenvs\.venv\Lib\site-packages\sklearn\metrics\


LogisticRegression
[[   0 1023    0]
 [   0 2479    1]
 [   0 1497    0]]
              precision    recall  f1-score   support

        High       0.00      0.00      0.00      1023
         Low       0.50      1.00      0.66      2480
      Medium       0.00      0.00      0.00      1497

    accuracy                           0.50      5000
   macro avg       0.17      0.33      0.22      5000
weighted avg       0.25      0.50      0.33      5000


DecisionTree
[[  52  773  198]
 [ 132 1955  393]
 [  84 1146  267]]
              precision    recall  f1-score   support

        High       0.19      0.05      0.08      1023
         Low       0.50      0.79      0.62      2480
      Medium       0.31      0.18      0.23      1497

    accuracy                           0.45      5000
   macro avg       0.34      0.34      0.31      5000
weighted avg       0.38      0.45      0.39      5000


RandomForest
[[  22  770  231]
 [  55 1938  487]
 [  35 1136  326]]
              precision  

,Model,Accuracy,Macro_F1
0,LogisticRegression,0.4958,0.220974
1,DecisionTree,0.4548,0.307557
2,RandomForest,0.4572,0.302754
3,GradientBoosting,0.4852,0.278124


### Step 6 — Select Best Model

In [7]:

best_model = RandomForestClassifier(n_estimators=200, random_state=42)
best_model.fit(X_train, y_train)

joblib.dump(best_model, "risk_model.pkl",compress=3)

print("Saved best risk model")


Saved best risk model


## MODEL B — Claim Outcome Classification

### Step 7 — Feature Selection

In [8]:

claim_features = [
    "billed_amount",
    "approved_amount",
    "payment_days",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "age",
    "chronic_flag"
]

X_train_c = train_df[claim_features].fillna(0)
X_test_c = test_df[claim_features].fillna(0)

y_train_c = train_df["claim_status"]
y_test_c = test_df["claim_status"]


### Step 8 — Train Multiple Models

In [9]:

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=10),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingClassifier()
}

results = []

for name, model in models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    
    acc = accuracy_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds, average="macro")
    
    results.append([name, acc, f1])
    
    print(f"\n{name}")
    print(confusion_matrix(y_test_c, preds))
    print(classification_report(y_test_c, preds))

results_df = pd.DataFrame(results, columns=["Model","Accuracy","Macro_F1"])
results_df


c:\Users\praka\.virtualenvs\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



LogisticRegression
[[2829    0  168]
 [ 764  449   62]
 [   0    0  728]]
              precision    recall  f1-score   support

        Paid       0.79      0.94      0.86      2997
     Pending       1.00      0.35      0.52      1275
    Rejected       0.76      1.00      0.86       728

    accuracy                           0.80      5000
   macro avg       0.85      0.77      0.75      5000
weighted avg       0.84      0.80      0.77      5000


DecisionTree
[[2820   65  112]
 [ 195 1029   51]
 [  39    8  681]]
              precision    recall  f1-score   support

        Paid       0.92      0.94      0.93      2997
     Pending       0.93      0.81      0.87      1275
    Rejected       0.81      0.94      0.87       728

    accuracy                           0.91      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.91      0.91      0.91      5000


RandomForest
[[2842   27  128]
 [ 113 1109   53]
 [  24    1  703]]
              precision  

,Model,Accuracy,Macro_F1
0,LogisticRegression,0.8012,0.747679
1,DecisionTree,0.9060,0.888096
2,RandomForest,0.9308,0.914305
3,GradientBoosting,0.9262,0.908779


### Step 9 — Train Final Model and Save

In [10]:

final_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_train_c, y_train_c)

joblib.dump(final_model, "claim_model.pkl",compress=3)

print("Saved best claim model")


Saved best claim model



## Step 10 — Feature Schema Export

The schema is exported to ensure production pipelines use the same features.


In [11]:

import json

schema = {
    "risk_model_features": risk_features,
    "claim_model_features": claim_features
}

with open("feature_schema.json","w") as f:
    json.dump(schema, f, indent=4)

print("Feature schema saved")


Feature schema saved
